In [1]:
import asammdf          # use asammdf to extract samples & timestamps
import numpy as np      # check compression OK using np.allclose
from io import BytesIO  # 
import sys
sys.path.append('../')
from mdfc import (
    MDFCompressor, MDFDecompressor
)

In [2]:
# parameters to generate sample MF4 file
example_mdf_params = dict(
    # random int/float data
    include_random_ints=False,
    include_random_floats=False,
    # non-random int/float data
    include_sine_waves=True,
    include_keepalives=True,
    # extent & grouping parameters
    time_s=3600,
    sample_intervals_ms=(1000,500,250,100),
    channels_per_interval=50,
    channels_per_group=4,
)

In [3]:
# uncompressed MDF file
from sample_data.generate_sample_data import generate_sample_file
MDF_FIL = BytesIO()
generate_sample_file(
    MDF_FIL, 
    compression=False,
    **example_mdf_params
)
MDF_FIL.seek(0); pass

96 total groups are generated


In [4]:
# size of uncompressed MDF file in MB
uncompressed_mdf_total_size = MDF_FIL.__sizeof__()
print(
    f'{uncompressed_mdf_total_size/1000/1000:.2f} '
    'MB Uncompressed MDF File'
)

48.02 MB Uncompressed MDF File


In [5]:
# comparison against using deflate, 
# (using asammdf parameter compression=1)
DEFLATE_MDF_FIL = BytesIO()
generate_sample_file(
    DEFLATE_MDF_FIL, 
    compression=1,
    **example_mdf_params
)
DEFLATE_MDF_FIL.seek(0); pass

96 total groups are generated


In [6]:
# size of deflated MDF file in MB
deflate_mdf_total_size = DEFLATE_MDF_FIL.__sizeof__()
print(
    f'{deflate_mdf_total_size/1000/1000:.2f} '
    'MB Deflate MDF File'
)

28.03 MB Deflate MDF File


In [7]:
# ratio of deflate vs uncompressed
print(
    f'{uncompressed_mdf_total_size/deflate_mdf_total_size:.3f} '
    'CR using Deflate (MDF Standard)'
)

1.713 CR using Deflate (MDF Standard)


In [8]:
# configurable parameters for mdfc compression,
# which is just for lossy float compression
# for lossless fp compression, set:
#   tolerance, significands, minimum_tolerance
#   = -1  (the default values)
#   TODO allow some false-y value also, or None
#        but presently, it would raise value error :(
# in this example we can use these lossy params:
lossy_fp_params = dict(
    significands = 3,
    # ^ meaning: 
    #   3 additional digits
    #   after the significance
    #   of the smallest value
    #       uniquely for each channel
    #   eg: 
    #       if channel A min_value == 1e-5,
    #       that channel tolerance =  1e-8
    minimum_tolerance = 1e-3,
    # ^ meaning:
    #   minimum tolerance for all channels
)

In [9]:
# test params
DO_TEST_COMPRESSION   = True
DO_TEST_DECOMPRESSION = True

In [10]:
# %%timeit
# test compression
if DO_TEST_COMPRESSION:
    MDFC_FIL = BytesIO()
    MDFC_FIL.seek(0); MDF_FIL.seek(0); pass
    with (
        asammdf.MDF(MDF_FIL) as mdf_fil,
        MDFCompressor(MDFC_FIL, close_file_on_exit=False) as mdfc_fil
    ):
        mdfc_fil.compress_all_signals(
            mdf_fil,
            on_error='warn',
            **lossy_fp_params,
        )
        # presently, must call finish function,
        #   TODO it should be done on a (successful?) __exit__
        mdfc_fil.finish()
    MDFC_FIL.seek(0); MDF_FIL.seek(0); pass

In [11]:
# size of mdfc file, MB
mdfc_total_size = MDFC_FIL.__sizeof__()
print(
    f'{mdfc_total_size/1000/1000:.2f} '
    'MB MDFC File'
)

5.41 MB MDFC File


In [12]:
# ratio of mdfc vs uncompressed
cr_vs_uncomp = (MDFC_FIL.__sizeof__() / MDF_FIL.__sizeof__())
print(
     'Overall compression ratio vs uncompressed is '
    f'{cr_vs_uncomp:.3f}, or {1/cr_vs_uncomp:.2f}x'
)

Overall compression ratio vs uncompressed is 0.113, or 8.87x


In [13]:
# ratio of mdfc vs deflate
cr_vs_deflate = (MDFC_FIL.__sizeof__() / DEFLATE_MDF_FIL.__sizeof__())
print(
     'Overall compression ratio vs Deflate is '
    f'{cr_vs_deflate:.3f}, or {1/cr_vs_deflate:.2f}x'
)

Overall compression ratio vs Deflate is 0.193, or 5.18x


In [14]:
# %%timeit
# execute decompression & compare against original
def decompress_and_compare(sn, mdfc_fil, mdf_fil):
    # decompress the signal from mdfc
    # and compare it against the signal in mdf
    original_sig = mdf_fil.select([sn], raw=True)[0]
    original_timestamps = original_sig.timestamps
    original_samples = original_sig.samples
    
    # decompress mdfc signal
    res = mdfc_fil.decompress_signal(sn)

    # assert all close timestamps and values
    # timestamps... may have some minor losses
    #   due to float->scaleup->int on compression
    #   i think it should be understood that the retention
    #   is on the order of +/- 1 nanosecond
    #       1e-10
    assert np.allclose(
        original_timestamps,
        res.timestamps,
        atol=1e-10  # precise at scale of nanoseconds
    ), f"{sn} timestamps not allclose!? :("
    assert np.allclose(
        original_samples,
        res.samples,
        # tolerance specification for float case
        # TODO perhaps this should be derived
        #   from compression metadata,
        #   ie the tolerance value used
        atol=lossy_fp_params['minimum_tolerance']
    ), f"{sn} samples not allclose!? :("


if DO_TEST_DECOMPRESSION:
    MDFC_FIL.seek(0); MDF_FIL.seek(0); pass
    with (
        asammdf.MDF(MDF_FIL) as mdf_fil,
        MDFDecompressor(MDFC_FIL, close_file_on_exit=False) as mdfc_fil
    ):
        try:
            # test signal decompression
            for sn in mdf_fil.channels_db.keys():
                if sn == 'time': continue  #
                decompress_and_compare(sn, mdfc_fil, mdf_fil)
        except KeyError:
            print(f'{sn} found in MDF but not in compressed file...')
            # raise  # ?
        else:
            print("All signals have passed decompression check :)")
    MDFC_FIL.seek(0); MDF_FIL.seek(0); pass

All signals have passed decompression check :)


In [15]:
# pass validity check :)

In [16]:
# lets do some time checks...
# test_names = [
#     ... specific signal names...
# ]
test_names = None  # all signals

In [17]:
%%timeit
# testing the speed of reading MDF (without compression)
MDF_FIL.seek(0)
with (
    asammdf.MDF(MDF_FIL) as mfil,
):
    if test_names is None:
        sigs = [
            sig_name # (sig_name, *chan_info) 
            for sig_name, chan_info in mfil.channels_db.items()
            if sig_name != 'time'
        ]
    else:
        sigs = test_names
    for sig_sel in sigs:
        sig = mfil.select([sig_sel], raw=True)[0]

109 ms ± 1.57 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [18]:
%%timeit
# testing the speed of reading MDF (with deflate compression)
DEFLATE_MDF_FIL.seek(0)
with (
    asammdf.MDF(DEFLATE_MDF_FIL) as mfil,
):
    if test_names is None:
        sigs = [
            sig_name # (sig_name, *chan_info) 
            for sig_name, chan_info in mfil.channels_db.items()
            if sig_name != 'time'
        ]
    else:
        sigs = test_names
    for sig_sel in sigs:
        sig = mfil.select([sig_sel], raw=True)[0]

432 ms ± 4.12 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [19]:
%%timeit
# testing the speed of reading the MDFC compressed file
MDFC_FIL.seek(0)
with MDFDecompressor(MDFC_FIL, close_file_on_exit=False) as dfil:
    if test_names is None:
        sigs = dfil.metadata.keys()
    else:
        sigs = test_names
    for sn in sigs:
        res = dfil.decompress_signal(sn)

161 ms ± 1.35 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [20]:
# end time checks :)

In [ ]:
# some old notes...

In [ ]:
# file size benchmarks:
# 98200 kb size uncompressed         MDF file
# 56300 kb size compressed (deflate) MDF file
# 40700 kb size compressed (zlib)    MDF file
#  9100 kb size MDFC file with "super-duper compression"
#   which is (fastpfor, or zfp) + zlib_9
#   and 1e-3 fp minimum_tolerance 
#       highly likely this is reached in this example data
#       it's a sinewave from -1 to 1 with some jitter
#       therefore, fp compression is highly variable 
#       to this user input tolerance
# so, between 5-10x additional compression vs MDF standard!
#   of course, with lossy floating-point compression

In [ ]:
# read time benchmarks:
#  250 ms to read the uncompressed         MDF file
# 1100 ms to read the compressed (deflate) MDF file
# 1160 ms to read the compressed (zlib)    MDF file
#  335 ms to read the MDFC file with "super-duper compression"

In [ ]:
# so the advantages of this MDF compression utility are:
# *) better compression than the ASAM standard (MDF + deflate)
#       and option for lossy fp compression
#       specifying a tolerance (comprehensible setting by engineers)
# *) faster to read than compressed MDF file
#       and can be similar time as reading uncompressed file!
#       although perhaps this is due to MDF file structure?
#       something about sorting/unsorting/etc?
# *) immediate access to time metadata,
#       likely better memory management when "normalizing to dataframe"
#       although that wrapper isnt implemented yet
#       and may not be too important?
# *) columnar access each signal
#       compared to "group-access",
#       which may add more time to compress the bytes packet
#           which might contain multiple signals in the same CAN msg


# the disadvantages:
# *) requires special code/libraries
#       although these can/should be precompiled?
#       which will therefore also allow 
#           some level of cross-platform
# *) would require update to go with update to ASAM standard
#       to support new unique types/shapes
# *) may require a lot of memory allocation for comp/decomp
#       if we always do one-shot compression
#       something like... 2*<longest_signal>*64 bits required
#           maybe 3*... 
#       if thats... 10 hours at 100ms per
#           thats ~50 MB required
#           call it 250 MB?
#           not suitable for embedded, 
#           but could be OK as a server-side utility
# *) cannot be seamlessly (/natively) integrated with existing tools
#       eg ETAS-MDA software
#       ...this is not a disadvantage, IMO. 
#       this is a server-side utility, not an analysis tool